# Session 4: Gradient Computation (2 hours)

**Goal:** Implement **full gradient descent for MSE** (loss, gradient, update loop). Then add an L2 penalty and sensemaking about what to check when it doesn't converge.

| Section | Time | Focus |
|---------|------|-------|
| Anchor case 1 | 50 min | Full GD for MSE (derivation, loop, L2 variation) |
| Anchor case 2 | 25 min | Numerical gradient check (dw vs finite diff) |
| Anchor case 3 | 30 min | Learning rate: too large vs too small (plot & describe) |
| Task 4.1 | 25 min | Derivatives & numerical verification |
| Task 4.2 | 45 min | 1D/2D GD, learning rate, LR for linear regression |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

---
## Anchor case: Full gradient descent for MSE [MEDIUM]

**Task:** We have loss(w) = mean((X @ w - y)²). Given X, y, write the gradient (math), then implement it in NumPy and run **100 steps** of gradient descent. Track loss each step.

**Gradient:** dw = (2/n) * X.T @ (X @ w - y). Shape: (d,).

### Worked example

In [ ]:
np.random.seed(42)
n, d = 100, 3
X = np.random.randn(n, d)
w_true = np.array([1.0, -0.5, 0.2])
y = X @ w_true + np.random.randn(n) * 0.3

w = np.zeros(d)
lr = 0.01
loss_history = []
for step in range(100):
    pred = X @ w
    loss = np.mean((pred - y) ** 2)
    loss_history.append(loss)
    error = pred - y   # (n,)
    dw = (2 / n) * (X.T @ error)  # (d,)
    w = w - lr * dw

print(f"Final w: {w}")
print(f"True w:  {w_true}")
print(f"Final loss: {loss_history[-1]:.6f}")
plt.plot(loss_history)
plt.xlabel('Step')
plt.ylabel('Loss (MSE)')
plt.title('Gradient descent for MSE')
plt.show()

### Your turn [MEDIUM]

1. Same setup with different n, d; run 100 steps and confirm loss goes down.
2. **L2 penalty:** Add λ * ||w||² to the loss (e.g. λ=0.01). The gradient of λ||w||² is 2λw. Update the gradient and run again.

In [ ]:
np.random.seed(0)
n, d = 50, 4
X = np.random.randn(n, d)
y = np.random.randn(n)
w = np.zeros(d)
lr = 0.01
lam = 0.01  # L2 strength
loss_history_l2 = []

for step in range(100):
    pred = X @ w
    loss = np.mean((pred - y) ** 2) + lam * np.sum(w ** 2)  # MSE + L2
    loss_history_l2.append(loss)
    error = pred - y
    dw_mse = (2 / n) * (X.T @ error)
    dw_l2 = 2 * lam * w
    dw = ...  # YOUR CODE: add both gradient parts
    w = w - lr * dw

plt.plot(loss_history_l2)
plt.xlabel('Step')
plt.ylabel('Loss (MSE + L2)')
plt.show()

<details>
<summary>💡 Click to see solution</summary>

```python
dw = dw_mse + dw_l2
```
</details>


### Sensemaking

**If loss doesn't go down, what are the first two things you'd check?** What does that tell you about your mental model?

_Your answer: e.g. (1) Learning rate too large (divergence) or too small (stuck). (2) Bug in gradient or sign (e.g. w = w - lr * dw). Checking shapes and a numerical gradient can confirm._

---
## Anchor case 2: Numerical gradient check for full GD [MEDIUM]

**Task:** Before trusting your gradient descent loop, verify **dw** (and optionally **db**) against a numerical gradient. Use the same idea as Session 3: perturb w_j by ±h, compute (loss(w+he_j) - loss(w-he_j))/(2h). Compare to your analytical dw; they should match to ~1e-5.

In [ ]:
def mse_loss(X, w, b, y):
    return np.mean((X @ w + b - y) ** 2)

def grad_w_analytical(X, w, b, y):
    n = len(y)
    return (2 / n) * (X.T @ (X @ w + b - y))

def grad_w_numerical(X, w, b, y, h=1e-6):
    dw = np.zeros_like(w)
    for j in range(len(w)):
        e = np.zeros_like(w); e[j] = 1
        dw[j] = (mse_loss(X, w + h*e, b, y) - mse_loss(X, w - h*e, b, y)) / (2 * h)
    return dw

np.random.seed(42)
n, d = 20, 3
X = np.random.randn(n, d)
w = np.random.randn(d)
b = 0.5
y = X @ w + b + np.random.randn(n) * 0.1
dw_ana = grad_w_analytical(X, w, b, y)
dw_num = grad_w_numerical(X, w, b, y)
print("Max |dw_ana - dw_num|:", np.abs(dw_ana - dw_num).max())

---
## Anchor case 3: Learning rate — too large vs too small [MEDIUM]

**Task:** Run gradient descent for MSE with (1) **lr = 0.5** and (2) **lr = 1e-5** for 100 steps each (use the same X, y, initial w). Plot loss vs step for both. Describe in one sentence what happens in each case (divergence? stuck?).

In [ ]:
np.random.seed(42)
n, d = 50, 3
X = np.random.randn(n, d)
w_true = np.array([1., -0.5, 0.2])
y = X @ w_true + np.random.randn(n) * 0.3

def run_gd(lr, steps=100):
    w = np.zeros(d)
    loss_hist = []
    for _ in range(steps):
        pred = X @ w
        loss_hist.append(np.mean((pred - y)**2))
        dw = (2/n) * (X.T @ (pred - y))
        w = w - lr * dw
    return loss_hist

loss_small_lr = run_gd(1e-5)
loss_large_lr = run_gd(0.5)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(loss_small_lr)
plt.title("lr = 1e-5")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.subplot(1, 2, 2)
plt.plot(loss_large_lr)
plt.title("lr = 0.5")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()
print("Describe: small lr = slow convergence; large lr = can diverge or oscillate.")

### Sensemaking

**If you had to pick a learning rate without plotting, what range would you try first (e.g. 1e-4 to 0.1)? Why?**

_Your answer: Often 1e-3 to 0.1 for well-scaled data. Too small wastes time; too large diverges. Normalising features helps._

---
## Task 4.1: Derivatives Review (25 min)

Implement analytical derivatives for key ML functions, then verify with numerical gradients.

### 4.1a: Implement Derivative Functions

In [ ]:
def f_squared(x):
    """f(x) = x^2"""
    return x ** 2

def derivative_squared(x):
    """d/dx of x^2 = 2x"""
    return 2 * x


def sigmoid(x):
    """sigmoid(x) = 1 / (1 + exp(-x))"""
    return 1 / (1 + np.exp(-x))

def derivative_sigmoid(x):
    """d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x))"""
    # TODO: Implement
    pass


def relu(x):
    """ReLU(x) = max(0, x)"""
    return np.maximum(0, x)

def derivative_relu(x):
    """d/dx ReLU(x) = 1 if x > 0, else 0"""
    # TODO: Implement
    pass


def mse(y_true, y_pred):
    """MSE = (1/n) * sum((y_true - y_pred)^2)"""
    return np.mean((y_true - y_pred) ** 2)

def derivative_mse(y_true, y_pred):
    """d(MSE)/d(y_pred) = (2/n) * (y_pred - y_true)"""
    # TODO: Implement
    pass

<details>
<summary>💡 Click to see solution</summary>

```python
def derivative_sigmoid(x):
    s = sigmoid(x)
    return s * (1 - s)

def derivative_relu(x):
    return (x > 0).astype(float)

def derivative_mse(y_true, y_pred):
    return 2 * (y_pred - y_true) / len(y_true)
```
</details>


### 4.1b: Numerical Gradient Verification

The numerical gradient approximates the derivative using the definition:

f'(x) ≈ [f(x + h) - f(x - h)] / (2h)

This is called the **central difference** and is more accurate than the forward difference.

In [ ]:
def numerical_gradient(f, x, h=1e-7):
    """Compute numerical gradient using central difference."""
    return (f(x + h) - f(x - h)) / (2 * h)


# Test at several points
test_points = np.array([-2.0, -1.0, 0.0, 0.5, 1.0, 2.0, 3.0])

print("=== f(x) = x^2 ===")
for x in test_points:
    analytical = derivative_squared(x)
    numerical = numerical_gradient(f_squared, x)
    print(f"  x={x:5.1f}: analytical={analytical:8.4f}, numerical={numerical:8.4f}, "
          f"match={np.isclose(analytical, numerical)}")

print("\n=== sigmoid(x) ===")
for x in test_points:
    analytical = derivative_sigmoid(x)
    numerical = numerical_gradient(sigmoid, x)
    print(f"  x={x:5.1f}: analytical={analytical:8.6f}, numerical={numerical:8.6f}, "
          f"match={np.isclose(analytical, numerical)}")

print("\n=== ReLU(x) ===")
for x in test_points:
    if x == 0:
        continue  # ReLU is not differentiable at 0
    analytical = derivative_relu(x)
    numerical = numerical_gradient(relu, x)
    print(f"  x={x:5.1f}: analytical={analytical:8.4f}, numerical={numerical:8.4f}, "
          f"match={np.isclose(analytical, numerical)}")

### 4.1c: Visualize the Functions and Their Derivatives

In [ ]:
x = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# x^2
axes[0].plot(x, f_squared(x), 'b-', linewidth=2, label='f(x) = x²')
axes[0].plot(x, derivative_squared(x), 'r--', linewidth=2, label="f'(x) = 2x")
axes[0].legend()
axes[0].set_title('Quadratic')
axes[0].grid(True, alpha=0.3)

# Sigmoid
axes[1].plot(x, sigmoid(x), 'b-', linewidth=2, label='σ(x)')
axes[1].plot(x, derivative_sigmoid(x), 'r--', linewidth=2, label="σ'(x)")
axes[1].legend()
axes[1].set_title('Sigmoid')
axes[1].grid(True, alpha=0.3)

# ReLU
axes[2].plot(x, relu(x), 'b-', linewidth=2, label='ReLU(x)')
axes[2].plot(x, derivative_relu(x), 'r--', linewidth=2, label="ReLU'(x)")
axes[2].legend()
axes[2].set_title('ReLU')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Task 4.2: Gradient Descent from Scratch (1.5 hours)

### 4.2a: 1D Gradient Descent on f(x) = x^2

Find the minimum of f(x) = x^2 starting from x = 10.

In [ ]:
def gradient_descent_1d():
    """Find minimum of f(x) = x^2 using gradient descent."""
    x = 10.0
    learning_rate = 0.1
    n_iterations = 50

    x_history = [x]
    f_history = [f_squared(x)]

    for i in range(n_iterations):
        # TODO: Compute gradient and update x
        # grad = ...
        # x = x - learning_rate * grad
        pass

        x_history.append(x)
        f_history.append(f_squared(x))

    return x_history, f_history


x_hist, f_hist = gradient_descent_1d()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trajectory on the function
x_plot = np.linspace(-11, 11, 100)
axes[0].plot(x_plot, f_squared(x_plot), 'b-', linewidth=2)
axes[0].plot(x_hist, f_hist, 'ro-', markersize=4, linewidth=1)
axes[0].plot(x_hist[0], f_hist[0], 'gs', markersize=12, label='Start')
axes[0].plot(x_hist[-1], f_hist[-1], 'r*', markersize=15, label='End')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('Gradient Descent on f(x) = x²')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: loss over iterations
axes[1].plot(f_hist, 'b-o', markersize=3)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('f(x)')
axes[1].set_title('Loss Over Iterations')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final x: {x_hist[-1]:.8f}")
print(f"Final f(x): {f_hist[-1]:.8f}")

<details>
<summary>💡 Click to see solution</summary>

```python
for i in range(n_iterations):
    grad = 2 * x
    x = x - learning_rate * grad
```
</details>


### 4.2b: 2D Gradient Descent on f(x, y) = x^2 + y^2

In [ ]:
def gradient_descent_2d():
    """Find minimum of f(x, y) = x^2 + y^2."""
    x, y = 8.0, 6.0
    learning_rate = 0.1
    n_iterations = 50

    history = [(x, y, x**2 + y**2)]

    for i in range(n_iterations):
        # TODO: Compute gradients and update x, y
        # df_dx = ...
        # df_dy = ...
        # x = x - learning_rate * df_dx
        # y = y - learning_rate * df_dy
        pass

        history.append((x, y, x**2 + y**2))

    return history


history = gradient_descent_2d()
xs = [h[0] for h in history]
ys = [h[1] for h in history]
fs = [h[2] for h in history]

# Contour plot showing the path
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: contour plot with path
grid = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(grid, grid)
Z = X**2 + Y**2

axes[0].contour(X, Y, Z, levels=20, cmap='viridis')
axes[0].plot(xs, ys, 'ro-', markersize=4, linewidth=1)
axes[0].plot(xs[0], ys[0], 'gs', markersize=12, label='Start')
axes[0].plot(xs[-1], ys[-1], 'r*', markersize=15, label='End')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Gradient Descent Path')
axes[0].legend()
axes[0].set_aspect('equal')

# Right: loss curve
axes[1].plot(fs, 'b-o', markersize=3)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('f(x, y)')
axes[1].set_title('Loss Over Iterations')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final (x, y): ({xs[-1]:.8f}, {ys[-1]:.8f})")
print(f"Final f(x, y): {fs[-1]:.8f}")

<details>
<summary>💡 Click to see solution</summary>

```python
for i in range(n_iterations):
    df_dx = 2 * x
    df_dy = 2 * y
    x = x - learning_rate * df_dx
    y = y - learning_rate * df_dy
```
</details>


### 4.2c: Learning Rate Exploration

What happens when the learning rate is too small? Too large?

In [ ]:
def gd_with_lr(learning_rate, n_iterations=50):
    """Run 1D GD on f(x)=x^2 with a given learning rate."""
    x = 10.0
    history = [x]
    for _ in range(n_iterations):
        grad = 2 * x
        x = x - learning_rate * grad
        history.append(x)
    return history


learning_rates = [0.01, 0.1, 0.5, 0.9, 1.0, 1.1]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

x_plot = np.linspace(-15, 15, 200)

for idx, lr in enumerate(learning_rates):
    hist = gd_with_lr(lr, n_iterations=20)
    f_hist = [h**2 for h in hist]

    axes[idx].plot(x_plot, x_plot**2, 'b-', linewidth=1, alpha=0.5)
    axes[idx].plot(hist, f_hist, 'ro-', markersize=4)
    axes[idx].plot(hist[0], f_hist[0], 'gs', markersize=10)
    axes[idx].set_title(f'LR = {lr}', fontsize=14)
    axes[idx].set_ylim(-5, 120)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### 4.2d: Gradient Descent for Linear Regression Cost

Apply gradient descent to minimize the MSE cost for a simple 1D linear regression.

In [ ]:
# Generate synthetic data: y = 3x + 7 + noise
np.random.seed(42)
n_samples = 50
X = np.random.rand(n_samples) * 10
y = 3 * X + 7 + np.random.randn(n_samples) * 2

plt.scatter(X, y, alpha=0.7)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Synthetic Data: y = 3x + 7 + noise')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def gradient_descent_linear(X, y, learning_rate=0.01, n_iterations=200):
    """
    Gradient descent for y = w*x + b.
    
    Cost = (1/2n) * sum((y_pred - y)^2)
    dw = (1/n) * sum((y_pred - y) * x)
    db = (1/n) * sum(y_pred - y)
    """
    w = 0.0
    b = 0.0
    n = len(X)
    history = {'w': [w], 'b': [b], 'cost': []}

    for i in range(n_iterations):
        # TODO: Forward pass
        # y_pred = ...

        # TODO: Compute cost
        # cost = ...

        # TODO: Compute gradients
        # dw = ...
        # db = ...

        # TODO: Update parameters
        # w = w - learning_rate * dw
        # b = b - learning_rate * db

        pass  # Remove this once you fill in the TODOs

        history['w'].append(w)
        history['b'].append(b)
        # history['cost'].append(cost)  # Uncomment after implementing

    return w, b, history


w, b, history = gradient_descent_linear(X, y, learning_rate=0.001, n_iterations=1000)
print(f"Learned: y = {w:.4f}x + {b:.4f}")
print(f"True:    y = 3.0000x + 7.0000")

<details>
<summary>💡 Click to see solution</summary>

```python
for i in range(n_iterations):
    y_pred = w * X + b
    cost = np.mean((y_pred - y) ** 2) / 2
    dw = np.mean((y_pred - y) * X)
    db = np.mean(y_pred - y)
    w = w - learning_rate * dw
    b = b - learning_rate * db
    history['cost'].append(cost)
```
</details>


In [ ]:
# Visualize the fit and loss history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: data + learned line
axes[0].scatter(X, y, alpha=0.7, label='Data')
x_line = np.linspace(X.min(), X.max(), 100)
axes[0].plot(x_line, w * x_line + b, 'r-', linewidth=2, label=f'Fit: y={w:.2f}x+{b:.2f}')
axes[0].plot(x_line, 3 * x_line + 7, 'g--', linewidth=2, label='True: y=3x+7', alpha=0.7)
axes[0].legend()
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')
axes[0].set_title('Linear Regression Fit')
axes[0].grid(True, alpha=0.3)

# Right: loss history
if history['cost']:
    axes[1].plot(history['cost'])
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Cost (MSE)')
    axes[1].set_title('Training Loss')
    axes[1].grid(True, alpha=0.3)
    plt.savefig('images/loss_history.png', dpi=150, bbox_inches='tight')
else:
    axes[1].text(0.5, 0.5, 'Implement cost tracking\nin gradient_descent_linear()',
                 ha='center', va='center', fontsize=12, transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

### Checkpoint

Can you explain why we multiply by `2/n_samples` (or `1/n_samples`) in the gradient?

**Your answer (double-click to edit):**

_Hint: Think about what the cost function is and how derivatives work with sums and constants..._

---
## Session 4 Reflection

**What happens when the learning rate is too large?**

_Your answer..._

**Why do we use the central difference for numerical gradients?**

_Your answer..._

**Connection:** The anchor case (MSE + gradient descent loop) is exactly what we put inside the `fit()` method in Session 5's `LinearRegressionScratch` class.